# [3단계 GeoAI] OSM 약지도 파인튜닝 — YOLO

제로샷 실패(정밀도 ~1%) 후속. OSM 철탑·변전소 좌표를 자동 라벨로 삼아
소형 탐지기(YOLOv8)를 파인튜닝한다.

- **학습 권역**: 화성·세종 / **평가 권역**: 서천·보령·은평 (OSM→OSM 순환 차단)
- 데이터셋은 로컬에서 만들어 HF `yolo_dataset.zip` 하나로 올려둠(약 1GB).
- **Kaggle: Accelerator=GPU T4, Internet=On** 필수.

## 순서
1. 셀 2에 `HF_TOKEN`·`HF_REPO` 입력 → Run All.
2. 학습(~1시간) → 평가 권역 추론 → 결과·가중치를 HF로 반출.
3. 로컬에서 `s3_hf_transfer.py` 변형으로 회수 후 OSM 대조.

In [ ]:
# 셀 1 — 설치
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음!')
%pip -q install ultralytics huggingface_hub --upgrade


In [ ]:
# 셀 2 — 설정 + 데이터셋 반입
HF_TOKEN = ''
HF_REPO  = ''   # 예 'jyxnq/wp2026-tiles-pilot'
EPOCHS, IMGSZ, BATCH = 80, 1024, 8

import os, zipfile
BASE = '/content' if os.path.isdir('/content') else '/kaggle/working'
DS = os.path.join(BASE, 'yolo')
from huggingface_hub import hf_hub_download
z = hf_hub_download(repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN or None,
                    filename='yolo_dataset.zip')
os.makedirs(DS, exist_ok=True)
zipfile.ZipFile(z).extractall(DS)
# data.yaml의 path를 절대경로로 고정
yml = os.path.join(DS, 'data.yaml')
txt = open(yml, encoding='utf-8').read().replace('path: .', f'path: {DS}')
open(yml, 'w', encoding='utf-8').write(txt)
print(open(yml, encoding='utf-8').read())
print('train 이미지', len(os.listdir(os.path.join(DS, 'train/images'))),
      '· val 이미지', len(os.listdir(os.path.join(DS, 'val/images'))))


In [ ]:
# 셀 3 — 학습 (YOLOv8s, 작은 객체라 imgsz=1024 유지)
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data=os.path.join(DS, 'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ,
            batch=BATCH, patience=20, project=os.path.join(BASE, 'runs'),
            name='pylon', exist_ok=True,
            # 작은 객체·약지도: 과한 증강 억제
            mosaic=0.5, scale=0.3, fliplr=0.5, degrees=0.0, translate=0.1)
print('학습 완료')


In [ ]:
# 셀 4 — 평가 권역 val 세트 지표 (YOLO 내장 — OSM 약라벨 기준 mAP/P/R)
best = os.path.join(BASE, 'runs', 'pylon', 'weights', 'best.pt')
m = YOLO(best)
metrics = m.val(data=os.path.join(DS, 'data.yaml'), imgsz=IMGSZ, split='val')
print('val mAP50:', round(float(metrics.box.map50), 3))
print('클래스별 P/R/mAP50:')
for i, name in enumerate(['tower', 'substation']):
    try:
        print(f'  {name}: P={metrics.box.p[i]:.3f} R={metrics.box.r[i]:.3f} '
              f'mAP50={metrics.box.ap50[i]:.3f}')
    except Exception:
        pass


In [ ]:
# 셀 5 — 가중치·학습로그를 HF로 반출 (로컬 회수용)
if HF_TOKEN and HF_REPO:
    from huggingface_hub import upload_file
    runs = os.path.join(BASE, 'runs', 'pylon')
    for rel in ['weights/best.pt', 'results.csv', 'confusion_matrix.png',
                'val_batch0_pred.jpg']:
        p = os.path.join(runs, rel)
        if os.path.isfile(p):
            upload_file(path_or_fileobj=p, path_in_repo=f'yolo_out/{os.path.basename(rel)}',
                        repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
            print('업로드:', rel)
    print('완료 — 로컬에서 best.pt 회수 후 전 타일 추론·OSM 대조')


## 다음 (로컬)
1. HF `yolo_out/best.pt` 회수.
2. 평가 권역 전 타일에 추론 → EPSG:5179 좌표화 → `s3_detect_vs_osm.py`로
   엄밀한 OSM 대조(매칭 반경 기준). 셀 4의 mAP는 약라벨 상자 기준 참고치이고,
   본 평가는 점-거리 매칭으로 로컬에서 한다.
3. 제로샷(재현율 철탑 4%) 대비 개선폭이 성패 판정.